# Cleaning the diabetes dataset for Apriori

Apriori needs a table of **True/False items** (one row = one patient stay, one column = one item like `insulin=Steady`).
So we will: load → clean → group values → turn numbers into categories → one-hot encode → save in `data/`.

## 1. Load the data

In [27]:
import os
import numpy as np
import pandas as pd

os.makedirs("Data", exist_ok=True)
df = pd.read_csv("/content/diabetic_data.csv", keep_default_na=False, na_values=["?"])
print(df.shape)
df.head()

(101766, 50)


/tmp/ipykernel_1963/815255750.py:6: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/diabetic_data.csv", keep_default_na=False, na_values=["?"])


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),NaN,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),NaN,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),NaN,1,1,7,2,...,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),NaN,1,1,7,1,...,No,Steady,No,No,No,No,No,Ch,Yes,NO


## 2. Look at the missing values

In [28]:
missing = df.isna().sum()
print(missing[missing > 0])

race                  2273
weight               98569
payer_code           40256
medical_specialty    49949
diag_1                  21
diag_2                 358
diag_3                1423
dtype: int64


## 3. Clean

**3.1 Drop columns with too many missing values, and the ID columns**

In [29]:
df = df.drop(columns=["weight", "payer_code", "encounter_id"])

**3.2 Keep only one row per patient**
(the same patient appears many times, which would distort the patterns)

In [30]:
df = df.drop_duplicates(subset="patient_nbr", keep="first")
df = df.drop(columns=["patient_nbr"])
print(df.shape)

(71518, 46)


**3.3 Remove invalid rows**

In [31]:
# gender not valid
df = df[df["gender"] != "Unknown/Invalid"]

# patients who died or went to hospice can not be readmitted (codes from IDS_mapping.csv)
died_codes = [11, 13, 14, 19, 20, 21]
df = df[~df["discharge_disposition_id"].isin(died_codes)]

# no diagnosis at all
df = df.dropna(subset=["diag_1"])
print(df.shape)

(69960, 46)


**3.4 Drop medicine columns that are almost always `No` (useless for patterns)**

In [32]:
drug_cols = ["metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride",
             "acetohexamide", "glipizide", "glyburide", "tolbutamide", "pioglitazone",
             "rosiglitazone", "acarbose", "miglitol", "troglitazone", "tolazamide", "examide",
             "citoglipton", "insulin", "glyburide-metformin", "glipizide-metformin",
             "glimepiride-pioglitazone", "metformin-rosiglitazone", "metformin-pioglitazone"]

kept_drugs = []
for drug in drug_cols:
    share_no = (df[drug] == "No").mean()      # share of rows with "No"
    if share_no >= 0.995:
        df = df.drop(columns=[drug])          # almost always "No" -> remove
    else:
        kept_drugs.append(drug)

print("Drugs kept:", kept_drugs)

Drugs kept: ['metformin', 'repaglinide', 'nateglinide', 'glimepiride', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'insulin', 'glyburide-metformin']


## 4. Group the ID columns (using `IDS_mapping.csv`)

In [33]:
# Codes that mean NULL / Not Mapped / Unknown are not in the dictionaries -> they become NaN (missing)
admission_type_dict = {1: "Emergency", 2: "Emergency", 7: "Emergency",
                       3: "Elective", 4: "Newborn"}

discharge_dict = {1: "Home", 6: "Home", 8: "Home",
                  2: "Transfer", 3: "Transfer", 4: "Transfer", 5: "Transfer", 15: "Transfer",
                  16: "Transfer", 17: "Transfer", 22: "Transfer", 23: "Transfer", 24: "Transfer",
                  27: "Transfer", 28: "Transfer", 29: "Transfer", 30: "Transfer",
                  7: "LeftAMA"}

admission_source_dict = {1: "Referral", 2: "Referral", 3: "Referral",
                         4: "Transfer", 5: "Transfer", 6: "Transfer", 10: "Transfer",
                         18: "Transfer", 22: "Transfer", 25: "Transfer", 26: "Transfer",
                         7: "EmergencyRoom",
                         11: "Birth", 12: "Birth", 13: "Birth", 14: "Birth", 23: "Birth", 24: "Birth"}

df["admission_type"]   = df["admission_type_id"].map(admission_type_dict)
df["discharge"]        = df["discharge_disposition_id"].map(discharge_dict)
df["admission_source"] = df["admission_source_id"].map(admission_source_dict)

df = df.drop(columns=["admission_type_id", "discharge_disposition_id", "admission_source_id"])
df[["admission_type", "discharge", "admission_source"]].head()

,admission_type,discharge,admission_source
0,NaN,NaN,Referral
1,Emergency,Home,EmergencyRoom
2,Emergency,Home,EmergencyRoom
3,Emergency,Home,EmergencyRoom
4,Emergency,Home,EmergencyRoom


**Medical specialty:** keep the 10 most frequent, the others become `Other`.

In [34]:
top10 = df["medical_specialty"].value_counts().head(10).index

# only change rows that are not missing and not in the top 10
mask = df["medical_specialty"].notna() & ~df["medical_specialty"].isin(top10)
df.loc[mask, "medical_specialty"] = "Other"

df["medical_specialty"].value_counts()

,count
medical_specialty,
InternalMedicine,10640
Other,5481
Family/GeneralPractice,4978
Emergency/Trauma,4392
Cardiology,4206
Surgery-General,2205
Orthopedics,1128
Orthopedics-Reconstructive,1041
Radiologist,821


## 5. Group the diagnosis codes (ICD-9)
There are hundreds of codes, so we group them into 9 families.

In [35]:
def diagnosis_group(code):
    if pd.isna(code):
        return np.nan                      # keep missing as missing
    code = str(code)
    if code.startswith("V") or code.startswith("E"):
        return "Other"
    number = float(code)
    if 250 <= number < 251:
        return "Diabetes"
    elif 390 <= number <= 459 or number == 785:
        return "Circulatory"
    elif 460 <= number <= 519 or number == 786:
        return "Respiratory"
    elif 520 <= number <= 579 or number == 787:
        return "Digestive"
    elif 800 <= number <= 999:
        return "Injury"
    elif 710 <= number <= 739:
        return "Musculoskeletal"
    elif 580 <= number <= 629 or number == 788:
        return "Genitourinary"
    elif 140 <= number <= 239:
        return "Neoplasms"
    else:
        return "Other"

for col in ["diag_1", "diag_2", "diag_3"]:
    df[col] = df[col].apply(diagnosis_group)

df["diag_1"].value_counts()

,count
diag_1,
Circulatory,21383
Other,12122
Respiratory,9486
Digestive,6487
Diabetes,5748
Injury,4692
Musculoskeletal,4064
Genitourinary,3440
Neoplasms,2538


## 6. Turn numbers into categories
Apriori can not use numbers, so we cut them into groups with `pd.cut`.

In [36]:
df["time_in_hospital"] = pd.cut(df["time_in_hospital"], bins=[0, 3, 7, 100],
                                labels=["short(1-3d)", "medium(4-7d)", "long(8d+)"]).astype(object)

df["num_lab_procedures"] = pd.cut(df["num_lab_procedures"], bins=[-1, 30, 50, 1000],
                                  labels=["low", "medium", "high"]).astype(object)

df["num_procedures"] = pd.cut(df["num_procedures"], bins=[-1, 0, 2, 100],
                              labels=["none", "few(1-2)", "many(3+)"]).astype(object)

df["num_medications"] = pd.cut(df["num_medications"], bins=[-1, 10, 20, 1000],
                               labels=["low(<=10)", "medium(11-20)", "high(21+)"]).astype(object)

df["number_diagnoses"] = pd.cut(df["number_diagnoses"], bins=[-1, 5, 8, 100],
                                labels=["few(<=5)", "moderate(6-8)", "many(9+)"]).astype(object)

# previous visits: 0 / 1 / 2 or more
for col in ["number_outpatient", "number_emergency", "number_inpatient"]:
    df[col] = pd.cut(df[col], bins=[-1, 0, 1, 1000],
                     labels=["none", "one", "several(2+)"]).astype(object)

df[["time_in_hospital", "num_medications"]].head()

,time_in_hospital,num_medications
0,short(1-3d),low(<=10)
1,short(1-3d),medium(11-20)
2,short(1-3d),medium(11-20)
3,short(1-3d),medium(11-20)
4,short(1-3d),low(<=10)


## 7. Remove useless items
- Drug value `No` = not prescribed → make it missing (no item).
- `None` = test not done → make it missing (no item).

In [37]:
for drug in kept_drugs:
    df[drug] = df[drug].replace("No", np.nan)

df["A1Cresult"]     = df["A1Cresult"].replace("None", np.nan)
df["max_glu_serum"] = df["max_glu_serum"].replace("None", np.nan)

df.head()

,race,gender,age,time_in_hospital,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,...,pioglitazone,rosiglitazone,insulin,glyburide-metformin,change,diabetesMed,readmitted,admission_type,discharge,admission_source
0,Caucasian,Female,[0-10),short(1-3d),Other,medium,none,low(<=10),none,none,...,NaN,NaN,NaN,NaN,No,No,NO,NaN,NaN,Referral
1,Caucasian,Female,[10-20),short(1-3d),NaN,high,none,medium(11-20),none,none,...,NaN,NaN,Up,NaN,Ch,Yes,>30,Emergency,Home,EmergencyRoom
2,AfricanAmerican,Female,[20-30),short(1-3d),NaN,low,many(3+),medium(11-20),several(2+),none,...,NaN,NaN,NaN,NaN,No,Yes,NO,Emergency,Home,EmergencyRoom
3,Caucasian,Male,[30-40),short(1-3d),NaN,medium,few(1-2),medium(11-20),none,none,...,NaN,NaN,Up,NaN,Ch,Yes,NO,Emergency,Home,EmergencyRoom
4,Caucasian,Male,[40-50),short(1-3d),NaN,high,none,low(<=10),none,none,...,NaN,NaN,Steady,NaN,Ch,Yes,NO,Emergency,Home,EmergencyRoom


## 8. One-hot encoding (True/False table for Apriori)

In [ ]:

onehot = pd.get_dummies(df, prefix_sep="=", dtype=int)

print(onehot.shape)
onehot.head()

(69960, 132)


,race=AfricanAmerican,race=Asian,race=Caucasian,race=Hispanic,race=Other,gender=Female,gender=Male,age=[0-10),age=[10-20),age=[20-30),...,admission_type=Elective,admission_type=Emergency,admission_type=Newborn,discharge=Home,discharge=LeftAMA,discharge=Transfer,admission_source=Birth,admission_source=EmergencyRoom,admission_source=Referral,admission_source=Transfer
0,0,0,1,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,1,0
1,0,0,1,0,0,1,0,0,1,0,...,0,1,0,1,0,0,0,1,0,0
2,1,0,0,0,0,1,0,0,0,1,...,0,1,0,1,0,0,0,1,0,0
3,0,0,1,0,0,0,1,0,0,0,...,0,1,0,1,0,0,0,1,0,0
4,0,0,1,0,0,0,1,0,0,0,...,0,1,0,1,0,0,0,1,0,0


## 9. Save the files in the `data/` folder

In [39]:
df.to_csv("diabetic_data_cleaned.csv", index=False)       # readable version
onehot.to_csv("diabetic_data_apriori.csv", index=False)   # ready for Apriori

print("Saved!", df.shape, onehot.shape)

Saved! (69960, 33) (69960, 132)
